# 🍽️ NutriSnap on Colab

**Part A: run the app now (~3 min, no GPU needed).** Run the 3 cells below; click the `https://….gradio.live` link that appears. It works on your phone too (camera supported).

**Part B: train the full model on a GPU** (Runtime → Change runtime type → T4 GPU), then relaunch the app with it.

## A1 · Get the code

In [ ]:
!git clone -q https://github.com/harshskapadia/Calories-.git /content/nutrisnap 2>/dev/null || git -C /content/nutrisnap pull -q
%cd /content/nutrisnap
!git log --oneline -1

## A2 · Install dependencies (~1–2 min; Colab already has PyTorch)

In [ ]:
!pip install -q timm gradio opencv-python-headless pyyaml
!python -c "import torch, timm, gradio; print('torch', torch.__version__, '| timm', timm.__version__, '| gradio', gradio.__version__, '| GPU:', torch.cuda.is_available())"

## A3 · Launch the app
Uses `app/model/nutrisnap.pt` from the repo. A public `gradio.live` link is printed below. Stop the cell to shut it down.

> ℹ️ If `app/model/nutrisnap.pt` isn't in the repo yet, run Part B first (or `python run_all.py` on your own GPU and push the model).

In [ ]:
!GRADIO_SHARE=1 python app/app.py

---
## Part B · Train on all 101 dishes with a GPU, then launch
Switch to a **GPU runtime** first, then re-run A1–A2.

### B1 · (Optional) Keep results in Google Drive
Colab wipes `/content` when the session ends. This stores checkpoints and the prepared dataset in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
P = '/content/drive/MyDrive/nutrisnap'; os.makedirs(f'{P}/checkpoints', exist_ok=True)
!rm -rf checkpoints && ln -s {P}/checkpoints checkpoints
# restore the prepared dataset if it was cached before
!test -f {P}/food101_256.tar && mkdir -p data/cache && tar -xf {P}/food101_256.tar -C data/cache && echo restored dataset from Drive || echo no cached dataset

### B2 · Download Food-101 from Hugging Face and pre-resize (~15–20 min, once)

In [ ]:
!pip install -q datasets scikit-learn seaborn
!python data/prepare_food101.py --config configs/full_gpu.yaml --workers 2
# cache to Drive for next time (only if B1 was run)
import os
if os.path.isdir('/content/drive/MyDrive/nutrisnap') and not os.path.exists('/content/drive/MyDrive/nutrisnap/food101_256.tar'):
    !tar -cf /content/drive/MyDrive/nutrisnap/food101_256.tar -C data/cache food101_256

### B3 · Train
Pick one:
- **Fast:** EfficientNet-B0, all 101 classes, full data, 6 epochs (~45 min on T4)
- **Best:** ConvNeXt-Tiny, 12 epochs (~2 h on T4, highest accuracy)

In [ ]:
EXPERIMENT = 'gpu_ablation_effnet_b0'   # fast
CONFIG = 'configs/gpu/ablation_effnet_b0.yaml'
# EXPERIMENT, CONFIG = 'full_convnext_tiny', 'configs/full_gpu.yaml'   # best
!python -m src.train --config {CONFIG} --set train.num_workers=2

### B4 · Evaluate on the official test split (25,250 images)

In [ ]:
!python -m src.evaluate --checkpoint checkpoints/{EXPERIMENT}/best.pt --num-workers 2
from IPython.display import Image, display
display(Image(f'checkpoints/{EXPERIMENT}/curves.png'))

### B5 · Launch the app with your new model

In [ ]:
!python scripts/export_model.py checkpoints/{EXPERIMENT}/best.pt --out app/model/nutrisnap.pt
!GRADIO_SHARE=1 python app/app.py

### B6 · Download the model (to put it in the GitHub repo / a Hugging Face Space)

In [ ]:
from google.colab import files
files.download('app/model/nutrisnap.pt')